# CycleGAN — Face ↔ Sketch Translation

**End-to-end notebook: dataset inspection → training → live loss plots → sample previews → checkpoint management → inference → training history**


> **Dataset path:** `./dataset/trainA`, `trainB`, `testA`, `testB`  
> Run `setup_dataset.py` first if you haven't already.

## Section 0 · Imports & GPU Check

In [8]:
import os, sys, time, random, itertools, glob
from pathlib import Path
from IPython.display import display, clear_output

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from torchvision.utils import make_grid
import torchvision.transforms as T

# ── Project files (must be in same folder as this notebook)
from models  import Generator, Discriminator, weights_init
from dataset import build_dataloaders, ImageBuffer

# ── RTX 40-series optimisations
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32        = True
torch.backends.cudnn.benchmark         = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('='*50)
print(f'  Device : {DEVICE}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  GPU    : {p.name}')
    print(f'  VRAM   : {p.total_memory/1e9:.1f} GB')
    print(f'  CUDA   : {torch.version.cuda}')
print(f'  PyTorch: {torch.__version__}')
print('='*50)

  Device : cpu
  PyTorch: 2.11.0+cpu


In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.version.cuda)

False
0
None


## Section 1 · Dataset Inspection

In [ ]:
# CONFIG
DATA_ROOT      = './dataset'
CHECKPOINT_DIR = './checkpoints'
SAMPLE_DIR     = './samples'
IMAGE_SIZE     = 256
BATCH_SIZE     = 2       # safe for RTX 4060 8 GB at 256px
EPOCHS         = 200
LR             = 2e-4
LAMBDA_CYC     = 10.0
LAMBDA_ID      = 5.0
NGF = NDF      = 64
N_RES          = 9
BUFFER_SIZE    = 50
SAVE_EVERY     = 1       # save checkpoint every N epochs
SAMPLE_EVERY   = 200     # save sample grid every N iterations
NUM_WORKERS    = 0       # MUST be 0 on Windows

Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(SAMPLE_DIR).mkdir(parents=True, exist_ok=True)
print('Directories ready ✓')

In [ ]:
# Count images per split
IMG_EXT = {'.jpg','.jpeg','.png','.bmp','.webp'}

def count_imgs(folder):
    p = Path(folder)
    if not p.exists(): return 0
    return sum(1 for f in p.rglob('*') if f.suffix.lower() in IMG_EXT)

splits = ['trainA','trainB','testA','testB']
counts = {s: count_imgs(f'{DATA_ROOT}/{s}') for s in splits}

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(splits, counts.values(),
              color=['#3b82f6','#22c55e','#f59e0b','#ef4444'],
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Images per split', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Count')
ax.spines[['top','right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

for s, c in counts.items():
    print(f'  {s:8s}: {c:>7,} images')

In [ ]:
# Show random face / sketch pairs
def load_random(folder, n=6):
    files = [f for f in Path(folder).rglob('*') if f.suffix.lower() in IMG_EXT]
    chosen = random.sample(files, min(n, len(files)))
    return [Image.open(p).convert('RGB').resize((200,200)) for p in chosen]

faces   = load_random(f'{DATA_ROOT}/trainA', 6)
sketches = load_random(f'{DATA_ROOT}/trainB', 6)

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('Dataset Samples — Top: Real Faces  |  Bottom: Sketches',
             fontsize=13, fontweight='bold', y=1.02)

for col, (face, sketch) in enumerate(zip(faces, sketches)):
    axes[0][col].imshow(face)
    axes[0][col].set_title(f'Face {col+1}', fontsize=9)
    axes[0][col].axis('off')
    axes[1][col].imshow(sketch)
    axes[1][col].set_title(f'Sketch {col+1}', fontsize=9)
    axes[1][col].axis('off')

plt.tight_layout()
plt.show()
print('These are UNPAIRED — CycleGAN does not need matching pairs ✓')

In [ ]:
# Pixel statistics — verify normalisation range
def pixel_stats(folder, n_samples=200):
    files = [f for f in Path(folder).rglob('*') if f.suffix.lower() in IMG_EXT]
    files = random.sample(files, min(n_samples, len(files)))
    means, stds = [], []
    for f in files:
        arr = np.array(Image.open(f).convert('RGB'), dtype=np.float32) / 255.0
        means.append(arr.mean(axis=(0,1)))
        stds.append(arr.std(axis=(0,1)))
    return np.mean(means, axis=0), np.mean(stds, axis=0)

print('Computing pixel statistics (sample of 200 images)...')
face_mean, face_std     = pixel_stats(f'{DATA_ROOT}/trainA')
sketch_mean, sketch_std = pixel_stats(f'{DATA_ROOT}/trainB')

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
channels = ['R','G','B']
x = np.arange(3)
w = 0.35

axes[0].bar(x-w/2, face_mean,   w, label='Faces',   color='#3b82f6', alpha=0.85)
axes[0].bar(x+w/2, sketch_mean, w, label='Sketches', color='#22c55e', alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(channels)
axes[0].set_title('Channel Mean (0–1)'); axes[0].legend()
axes[0].set_ylim(0,1); axes[0].spines[['top','right']].set_visible(False)

axes[1].bar(x-w/2, face_std,   w, label='Faces',   color='#3b82f6', alpha=0.85)
axes[1].bar(x+w/2, sketch_std, w, label='Sketches', color='#22c55e', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(channels)
axes[1].set_title('Channel Std (0–1)'); axes[1].legend()
axes[1].set_ylim(0,0.5); axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Pixel Statistics', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Faces   — mean: {face_mean}  std: {face_std}')
print(f'Sketches — mean: {sketch_mean}  std: {sketch_std}')
print('Normalise to [-1,1] using mean=0.5, std=0.5 ✓')

## Section 2 · Build Models

In [ ]:
# Instantiate all 4 networks
G_AB = Generator(ngf=NGF, n_res_blocks=N_RES).to(DEVICE)  # face  → sketch
G_BA = Generator(ngf=NGF, n_res_blocks=N_RES).to(DEVICE)  # sketch → face
D_A  = Discriminator(ndf=NDF).to(DEVICE)                   # judge faces
D_B  = Discriminator(ndf=NDF).to(DEVICE)                   # judge sketches

G_AB.apply(weights_init)
G_BA.apply(weights_init)
D_A.apply(weights_init)
D_B.apply(weights_init)

def count_params(m):
    return sum(p.numel() for p in m.parameters()) / 1e6

print('Model Summary')
print('─'*40)
print(f'  G_AB (face → sketch) : {count_params(G_AB):.2f} M params')
print(f'  G_BA (sketch → face) : {count_params(G_BA):.2f} M params')
print(f'  D_A  (face disc)     : {count_params(D_A):.2f} M params')
print(f'  D_B  (sketch disc)   : {count_params(D_B):.2f} M params')
print('─'*40)
total = count_params(G_AB)+count_params(G_BA)+count_params(D_A)+count_params(D_B)
print(f'  Total                : {total:.2f} M params')

# Quick forward pass sanity check
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
with torch.no_grad():
    out_g = G_AB(dummy)
    out_d = D_A(dummy)
print(f'\nSanity check ✓')
print(f'  Generator  : {list(dummy.shape)} → {list(out_g.shape)}')
print(f'  Discriminator: {list(dummy.shape)} → {list(out_d.shape)}')

In [ ]:
# Losses, Optimisers, Schedulers, AMP
criterion_GAN   = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_id    = nn.L1Loss()

n_epochs_decay = EPOCHS // 2
n_epochs_const = EPOCHS - n_epochs_decay

opt_G   = torch.optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()),
                            lr=LR, betas=(0.5, 0.999))
opt_D_A = torch.optim.Adam(D_A.parameters(), lr=LR, betas=(0.5, 0.999))
opt_D_B = torch.optim.Adam(D_B.parameters(), lr=LR, betas=(0.5, 0.999))

def lr_lambda(epoch):
    if epoch < n_epochs_const: return 1.0
    return max(0.0, 1.0 - (epoch - n_epochs_const) / max(n_epochs_decay, 1))

sched_G   = LambdaLR(opt_G,   lr_lambda=lr_lambda)
sched_D_A = LambdaLR(opt_D_A, lr_lambda=lr_lambda)
sched_D_B = LambdaLR(opt_D_B, lr_lambda=lr_lambda)

USE_AMP = (DEVICE.type == 'cuda')
scaler_G   = torch.amp.GradScaler('cuda', enabled=USE_AMP)
scaler_D_A = torch.amp.GradScaler('cuda', enabled=USE_AMP)
scaler_D_B = torch.amp.GradScaler('cuda', enabled=USE_AMP)

buffer_A = ImageBuffer(BUFFER_SIZE)
buffer_B = ImageBuffer(BUFFER_SIZE)

def make_target(pred, real):
    return torch.full_like(pred, 1.0 if real else 0.0, device=DEVICE)

print('Optimisers, schedulers, AMP scalers ready ✓')
print(f'AMP (mixed precision): {"enabled" if USE_AMP else "disabled (CPU)"}')

In [ ]:
# Resume from checkpoint if one exists
START_EPOCH = 0
latest_ckpt = Path(CHECKPOINT_DIR) / 'latest.pth'

if latest_ckpt.exists():
    print(f'Found checkpoint: {latest_ckpt}')
    ckpt = torch.load(str(latest_ckpt), map_location=DEVICE)
    G_AB.load_state_dict(ckpt['G_AB'])
    G_BA.load_state_dict(ckpt['G_BA'])
    D_A.load_state_dict(ckpt['D_A'])
    D_B.load_state_dict(ckpt['D_B'])
    opt_G.load_state_dict(ckpt['opt_G'])
    opt_D_A.load_state_dict(ckpt['opt_D_A'])
    opt_D_B.load_state_dict(ckpt['opt_D_B'])
    START_EPOCH = ckpt['epoch'] + 1
    for _ in range(START_EPOCH):
        sched_G.step(); sched_D_A.step(); sched_D_B.step()
    print(f'  Resumed from epoch {START_EPOCH} ✓')
    # Restore loss history if saved
    loss_history = ckpt.get('loss_history', {'G':[],'D_A':[],'D_B':[]})
else:
    print('No checkpoint found — starting fresh training.')
    loss_history = {'G': [], 'D_A': [], 'D_B': []}

## Section 3 · Training Loop with Live Loss Plots

In [ ]:
# Visualisation helpers
def tensor_to_img(t):
    """Single tensor [-1,1] → numpy HWC uint8."""
    t = (t.squeeze(0).detach().cpu() * 0.5 + 0.5).clamp(0,1)
    return (t.permute(1,2,0).numpy() * 255).astype(np.uint8)

def show_sample_grid(real_A, fake_B, rec_A, real_B, fake_A, rec_B, epoch, iteration):
    """Show 2-row grid: face-cycle (top) and sketch-cycle (bottom)."""
    with torch.no_grad():
        rows = [
            [tensor_to_img(real_A[0]), tensor_to_img(fake_B[0]), tensor_to_img(rec_A[0])],
            [tensor_to_img(real_B[0]), tensor_to_img(fake_A[0]), tensor_to_img(rec_B[0])],
        ]
    titles_top = ['Real Face', 'Fake Sketch (G_AB)', 'Reconstructed Face']
    titles_bot = ['Real Sketch', 'Fake Face (G_BA)', 'Reconstructed Sketch']

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    fig.suptitle(f'Epoch {epoch+1} | Iteration {iteration}',
                 fontsize=13, fontweight='bold')
    for col in range(3):
        axes[0][col].imshow(rows[0][col]); axes[0][col].set_title(titles_top[col])
        axes[1][col].imshow(rows[1][col]); axes[1][col].set_title(titles_bot[col])
        axes[0][col].axis('off'); axes[1][col].axis('off')

    # Draw arrows
    for col in [0, 1]:
        fig.text(0.34 + col*0.33, 0.52, '→', fontsize=22,
                 ha='center', color='#475569')
        fig.text(0.34 + col*0.33, 0.02, '→', fontsize=22,
                 ha='center', color='#475569')

    plt.tight_layout()
    save_path = f'{SAMPLE_DIR}/epoch{epoch+1:03d}_iter{iteration:05d}.png'
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

def plot_losses(history, epoch):
    """Live updating loss curves."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    labels  = ['Generator Loss', 'D_A Loss (Faces)', 'D_B Loss (Sketches)']
    keys    = ['G', 'D_A', 'D_B']
    colors  = ['#7c3aed', '#2563eb', '#16a34a']

    for ax, key, label, color in zip(axes, keys, labels, colors):
        data = history[key]
        if len(data) == 0:
            ax.text(0.5, 0.5, 'No data yet', ha='center', va='center',
                    transform=ax.transAxes, color='#9ca3af')
        else:
            ax.plot(data, color=color, linewidth=1.2, alpha=0.8)
            # Rolling mean
            if len(data) >= 10:
                rm = np.convolve(data, np.ones(10)/10, mode='valid')
                ax.plot(range(9, len(data)), rm, color=color,
                        linewidth=2.5, label='10-epoch avg')
                ax.legend(fontsize=8)
            ax.set_title(label, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Loss')
            ax.spines[['top','right']].set_visible(False)
            ax.yaxis.grid(True, alpha=0.3)
            ax.set_axisbelow(True)

    plt.suptitle(f'Training Losses — epoch {epoch+1}/{EPOCHS}',
                 fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{SAMPLE_DIR}/loss_curves.png', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()

print('Visualisation helpers ready ✓')

In [ ]:
# Build dataloaders
train_loader, test_loader = build_dataloaders(
    DATA_ROOT, IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS
)
total_iters = len(train_loader)
print(f'Train batches per epoch : {total_iters:,}')
print(f'Starting from epoch     : {START_EPOCH + 1}/{EPOCHS}')
print()

# Training loop
for epoch in range(START_EPOCH, EPOCHS):
    epoch_start = time.time()
    running = {'G': 0.0, 'D_A': 0.0, 'D_B': 0.0}

    G_AB.train(); G_BA.train(); D_A.train(); D_B.train()

    for i, batch in enumerate(train_loader):
        real_A = batch['A'].to(DEVICE)
        real_B = batch['B'].to(DEVICE)

        # (1) Update Generators
        for p in D_A.parameters(): p.requires_grad_(False)
        for p in D_B.parameters(): p.requires_grad_(False)
        opt_G.zero_grad()

        with torch.amp.autocast('cuda', enabled=USE_AMP):
            id_B = G_AB(real_B)
            loss_id_B = criterion_id(id_B, real_B) * LAMBDA_ID
            id_A = G_BA(real_A)
            loss_id_A = criterion_id(id_A, real_A) * LAMBDA_ID

            fake_B = G_AB(real_A)
            loss_GAN_AB = criterion_GAN(D_B(fake_B), make_target(D_B(fake_B), True))
            fake_A = G_BA(real_B)
            loss_GAN_BA = criterion_GAN(D_A(fake_A), make_target(D_A(fake_A), True))

            rec_A = G_BA(fake_B)
            loss_cyc_A = criterion_cycle(rec_A, real_A) * LAMBDA_CYC
            rec_B = G_AB(fake_A)
            loss_cyc_B = criterion_cycle(rec_B, real_B) * LAMBDA_CYC

            loss_G = (loss_GAN_AB + loss_GAN_BA
                      + loss_cyc_A + loss_cyc_B
                      + loss_id_A  + loss_id_B)

        scaler_G.scale(loss_G).backward()
        scaler_G.step(opt_G)
        scaler_G.update()

        # (2) Update D_A
        for p in D_A.parameters(): p.requires_grad_(True)
        opt_D_A.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            pred_real_A = D_A(real_A)
            fake_A_buf  = buffer_A.push_and_pop(fake_A.detach())
            pred_fake_A = D_A(fake_A_buf)
            loss_D_A = (criterion_GAN(pred_real_A, make_target(pred_real_A, True))
                      + criterion_GAN(pred_fake_A, make_target(pred_fake_A, False))) * 0.5
        scaler_D_A.scale(loss_D_A).backward()
        scaler_D_A.step(opt_D_A)
        scaler_D_A.update()

        # (3) Update D_B
        for p in D_B.parameters(): p.requires_grad_(True)
        opt_D_B.zero_grad()
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            pred_real_B = D_B(real_B)
            fake_B_buf  = buffer_B.push_and_pop(fake_B.detach())
            pred_fake_B = D_B(fake_B_buf)
            loss_D_B = (criterion_GAN(pred_real_B, make_target(pred_real_B, True))
                      + criterion_GAN(pred_fake_B, make_target(pred_fake_B, False))) * 0.5
        scaler_D_B.scale(loss_D_B).backward()
        scaler_D_B.step(opt_D_B)
        scaler_D_B.update()

        running['G']   += loss_G.item()
        running['D_A'] += loss_D_A.item()
        running['D_B'] += loss_D_B.item()

        # Inline sample grid every N iters
        if (i + 1) % SAMPLE_EVERY == 0:
            G_AB.eval(); G_BA.eval()
            with torch.no_grad():
                fb = G_AB(real_A); fa = G_BA(real_B)
                rb = G_AB(fa);     ra = G_BA(fb)
            clear_output(wait=True)
            show_sample_grid(real_A, fb, ra, real_B, fa, rb, epoch, i+1)
            G_AB.train(); G_BA.train()

        # Inline progress
        print(f'\r  Ep [{epoch+1:3d}/{EPOCHS}] '
              f'[{i+1:4d}/{total_iters}] '
              f'G:{loss_G.item():.3f} '
              f'D_A:{loss_D_A.item():.3f} '
              f'D_B:{loss_D_B.item():.3f}',
              end='', flush=True)

    # End of epoch
    elapsed = time.time() - epoch_start
    avg_G   = running['G']   / total_iters
    avg_D_A = running['D_A'] / total_iters
    avg_D_B = running['D_B'] / total_iters

    loss_history['G'].append(avg_G)
    loss_history['D_A'].append(avg_D_A)
    loss_history['D_B'].append(avg_D_B)

    print(f'\n  → Ep {epoch+1:3d} | {elapsed:.0f}s '
          f'| G={avg_G:.4f} D_A={avg_D_A:.4f} D_B={avg_D_B:.4f}')

    sched_G.step(); sched_D_A.step(); sched_D_B.step()

    # Live loss plot every epoch
    plot_losses(loss_history, epoch)

    # Checkpoint
    if (epoch + 1) % SAVE_EVERY == 0:
        state = {
            'epoch': epoch,
            'G_AB': G_AB.state_dict(), 'G_BA': G_BA.state_dict(),
            'D_A':  D_A.state_dict(),  'D_B':  D_B.state_dict(),
            'opt_G': opt_G.state_dict(),
            'opt_D_A': opt_D_A.state_dict(),
            'opt_D_B': opt_D_B.state_dict(),
            'loss_history': loss_history,   # persists loss curves across sessions
        }
        ckpt_path = f'{CHECKPOINT_DIR}/cyclegan_epoch{epoch+1:03d}.pth'
        torch.save(state, ckpt_path)
        torch.save(state, f'{CHECKPOINT_DIR}/latest.pth')
        print(f'  ✓ Checkpoint saved → {ckpt_path}')

print('\n\nTraining complete!')

## Section 4 · Checkpoint Browser

In [ ]:
# List all saved checkpoints
ckpts = sorted(Path(CHECKPOINT_DIR).glob('cyclegan_epoch*.pth'))

if not ckpts:
    print('No checkpoints yet. Train first (Section 3).')
else:
    print(f'Found {len(ckpts)} checkpoint(s):\n')
    rows = []
    for p in ckpts:
        size_mb = p.stat().st_size / 1e6
        ckpt = torch.load(str(p), map_location='cpu')
        ep   = ckpt.get('epoch', '?')
        hist = ckpt.get('loss_history', {})
        g_loss  = hist['G'][-1]   if hist.get('G')   else float('nan')
        da_loss = hist['D_A'][-1] if hist.get('D_A') else float('nan')
        rows.append((p.name, ep, g_loss, da_loss, size_mb))

    print(f'  {"File":35s}  {"Epoch":>6}  {"G loss":>8}  {"D_A loss":>9}  {"Size MB":>8}')
    print('  ' + '─'*75)
    for name, ep, g, da, sz in rows:
        print(f'  {name:35s}  {str(ep):>6}  {g:8.4f}  {da:9.4f}  {sz:8.1f}')

    latest = Path(CHECKPOINT_DIR) / 'latest.pth'
    if latest.exists():
        lc = torch.load(str(latest), map_location='cpu')
        print(f'\n  latest.pth → epoch {lc["epoch"]+1}')

In [ ]:
# Load a specific checkpoint
# Change this to load any epoch you want
LOAD_EPOCH = 'latest'   # e.g. 'latest' or 50 or 100

if LOAD_EPOCH == 'latest':
    ckpt_path = Path(CHECKPOINT_DIR) / 'latest.pth'
else:
    ckpt_path = Path(CHECKPOINT_DIR) / f'cyclegan_epoch{int(LOAD_EPOCH):03d}.pth'

if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location=DEVICE)
    G_AB.load_state_dict(ckpt['G_AB'])
    G_BA.load_state_dict(ckpt['G_BA'])
    G_AB.eval(); G_BA.eval()
    loaded_epoch = ckpt['epoch'] + 1
    print(f'✓ Loaded checkpoint from epoch {loaded_epoch}')

    # Re-plot loss history from this checkpoint
    if 'loss_history' in ckpt and ckpt['loss_history']['G']:
        plot_losses(ckpt['loss_history'], ckpt['epoch'])
else:
    print(f'Checkpoint not found: {ckpt_path}')

## Section 5 · Evaluation Grid

In [ ]:
# Evaluate on N test image pairs
N_EVAL = 8   # number of test images to show

transform_test = T.Compose([
    T.Resize(IMAGE_SIZE, Image.BICUBIC),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])

def load_test_imgs(folder, n):
    files = sorted([f for f in Path(folder).rglob('*')
                    if f.suffix.lower() in IMG_EXT])[:n]
    return [transform_test(Image.open(f).convert('RGB')).unsqueeze(0).to(DEVICE)
            for f in files]

test_faces   = load_test_imgs(f'{DATA_ROOT}/testA', N_EVAL)
test_sketches = load_test_imgs(f'{DATA_ROOT}/testB', N_EVAL)

G_AB.eval(); G_BA.eval()

fig, axes = plt.subplots(4, N_EVAL, figsize=(N_EVAL*2.5, 11))
row_labels = ['Real Face', 'Face → Sketch', 'Real Sketch', 'Sketch → Face']

for col in range(N_EVAL):
    with torch.no_grad():
        fake_sketch = G_AB(test_faces[col])
        fake_face   = G_BA(test_sketches[col])

    imgs = [
        tensor_to_img(test_faces[col]),
        tensor_to_img(fake_sketch),
        tensor_to_img(test_sketches[col]),
        tensor_to_img(fake_face),
    ]
    for row in range(4):
        axes[row][col].imshow(imgs[row])
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(row_labels[row], fontsize=9,
                                       fontweight='bold', rotation=90,
                                       labelpad=4)
            axes[row][col].yaxis.set_label_position('left')

    # Separator between face and sketch blocks
    if col == N_EVAL // 2 - 1:
        for row in range(4):
            axes[row][col].spines['right'].set_visible(True)
            axes[row][col].spines['right'].set_linewidth(2)
            axes[row][col].spines['right'].set_color('#94a3b8')

fig.suptitle(f'Evaluation Grid — Epoch {loaded_epoch}',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
eval_path = f'{SAMPLE_DIR}/eval_epoch{loaded_epoch:03d}.png'
plt.savefig(eval_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'\nSaved → {eval_path}')

In [ ]:
# Cycle-consistency check
# Shows: Real → Translated → Reconstructed (should look like Real)
N_CYCLE = 4

fig, axes = plt.subplots(2, N_CYCLE*3, figsize=(N_CYCLE*9, 6))
fig.suptitle('Cycle Consistency: A→B→A  and  B→A→B',
             fontsize=13, fontweight='bold')

for i in range(N_CYCLE):
    face   = test_faces[i]
    sketch = test_sketches[i]
    with torch.no_grad():
        fake_s = G_AB(face)
        rec_f  = G_BA(fake_s)
        fake_f = G_BA(sketch)
        rec_s  = G_AB(fake_f)

    base = i * 3
    for img, col, title in [
        (face,   base,   'Real Face'),
        (fake_s, base+1, '→ Sketch'),
        (rec_f,  base+2, '→ Recon Face'),
    ]:
        axes[0][col].imshow(tensor_to_img(img))
        axes[0][col].set_title(title, fontsize=8)
        axes[0][col].axis('off')

    for img, col, title in [
        (sketch, base,   'Real Sketch'),
        (fake_f, base+1, '→ Face'),
        (rec_s,  base+2, '→ Recon Sketch'),
    ]:
        axes[1][col].imshow(tensor_to_img(img))
        axes[1][col].set_title(title, fontsize=8)
        axes[1][col].axis('off')

plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/cycle_consistency_epoch{loaded_epoch:03d}.png',
            dpi=110, bbox_inches='tight')
plt.show()

## Section 6 · Single-Image Inference

In [ ]:
# Translate any image
# Set your image path here
IMAGE_PATH = './dataset/testA/5141.jpg'
DIRECTION  = 'auto'   # 'auto' | 'face2sketch' | 'sketch2face'

def detect_domain(image):
    arr = np.array(image.convert('RGB'), dtype=np.float32) / 255.0
    r,g,b = arr[...,0], arr[...,1], arr[...,2]
    cmax, cmin = arr.max(axis=-1), arr.min(axis=-1)
    sat = np.where(cmax > 0, (cmax-cmin)/(cmax+1e-8), 0.0)
    grey_std = (0.299*r + 0.587*g + 0.114*b).std()
    return 'sketch' if (sat.mean() < 0.12 and grey_std > 0.20) else 'face'

if not Path(IMAGE_PATH).exists():
    # Fallback: use first test image
    test_files = list(Path(f'{DATA_ROOT}/testA').rglob('*'))
    test_files = [f for f in test_files if f.suffix.lower() in IMG_EXT]
    if test_files:
        IMAGE_PATH = str(test_files[0])
        print(f'IMAGE_PATH not set — using: {IMAGE_PATH}')
    else:
        print('No test images found. Set IMAGE_PATH manually.')

img = Image.open(IMAGE_PATH).convert('RGB')
detected = detect_domain(img)

if DIRECTION == 'auto':
    direction = 'face2sketch' if detected == 'face' else 'sketch2face'
else:
    direction = DIRECTION

tensor = transform_test(img).unsqueeze(0).to(DEVICE)
G_AB.eval(); G_BA.eval()

with torch.no_grad():
    output = G_AB(tensor) if direction == 'face2sketch' else G_BA(tensor)

# Display
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(img.resize((256,256)))
ax1.set_title(f'Input ({detected})', fontsize=12, fontweight='bold')
ax1.axis('off')

label = 'Generated Sketch' if direction == 'face2sketch' else 'Generated Face'
ax2.imshow(tensor_to_img(output))
ax2.set_title(label, fontsize=12, fontweight='bold')
ax2.axis('off')

arrow_label = 'Face → Sketch  (G_AB)' if direction == 'face2sketch' else 'Sketch → Face  (G_BA)'
fig.suptitle(arrow_label, fontsize=13, fontweight='bold', color='#7c3aed')
plt.tight_layout()

out_path = str(Path(IMAGE_PATH).stem) + '_translated.png'
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')
print(f'Detected domain : {detected}')
print(f'Direction used  : {direction}')

## Section 7 · Training History

In [ ]:
# Full training history plot (run anytime)
ckpt = torch.load(str(Path(CHECKPOINT_DIR)/'latest.pth'), map_location='cpu')
hist = ckpt.get('loss_history', {'G':[],'D_A':[],'D_B':[]})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
titles = ['Generator Loss', 'Discriminator A (Faces)', 'Discriminator B (Sketches)']
keys   = ['G', 'D_A', 'D_B']
colors = ['#7c3aed', '#2563eb', '#16a34a']

for ax, key, title, color in zip(axes, keys, titles, colors):
    data = hist.get(key, [])
    epochs_x = range(1, len(data)+1)
    ax.plot(epochs_x, data, color=color, alpha=0.5, linewidth=1)
    if len(data) >= 5:
        rm = np.convolve(data, np.ones(5)/5, mode='valid')
        ax.plot(range(5, len(data)+1), rm, color=color, linewidth=2.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Avg Loss')
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.grid(True, alpha=0.3)
    ax.set_axisbelow(True)
    if data:
        ax.set_title(f'{title}\n(final: {data[-1]:.4f})', fontweight='bold', fontsize=10)

plt.suptitle(f'Complete Training History — {len(hist["G"])} epochs',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAMPLE_DIR}/full_loss_history.png', dpi=120, bbox_inches='tight')
plt.show()